# GDACS — Historical National Exposure

Retrieves country-level population exposure for all tropical cyclones within a
date range, using the GDACS search endpoint for full historical coverage.

**API path:**
```
geteventlist/search  (paginated, filterable by date / source)
  -> getepisodedata  (latest episode per event)
    -> getimpact -> datums[alias='country'] -> ISO_3DIGIT, CNTRY_NAME, POP_AFFECTED
```

**Source filter:**
| `source` | Basin |
|---|---|
| `NOAA` | Atlantic + Eastern Pacific |
| `JTWC` | Western Pacific + Indian Ocean |
| *(omit)* | All basins |

In [ ]:
import requests
import pandas as pd

GDACS_BASE = 'https://www.gdacs.org/gdacsapi/api'

# ── Date range ────────────────────────────────────────────────────────────────
FROM_DATE = '2010-01-01'
TO_DATE   = '2026-12-31'

# ── Basin filter — set to None to include all basins ─────────────────────────
SOURCE    = 'NOAA'   # 'NOAA' = Atlantic/E.Pacific | 'JTWC' = W.Pacific/Indian Ocean

# ── Output file ───────────────────────────────────────────────────────────────
OUTPUT_CSV = 'gdacs_historical_national_exposure.csv'

# Wind speed (kt) implied by each buffer key
BUFFER_KT = {
    'buffer39': 34,
    'buffer74': 64,
}

In [12]:
# ── Fetch all TC events in the date range via paginated search ────────────────
all_events = []
page = 1

while True:
    params = {
        'eventlist':  'TC',
        'fromDate':   FROM_DATE,
        'toDate':     TO_DATE,
        'pageSize':   100,
        'pageNumber': page,
    }
    resp = requests.get(
        f'{GDACS_BASE}/events/geteventlist/search',
        params=params,
        timeout=30,
    )
    # API returns an empty body (not JSON) when there are no more pages
    if not resp.text.strip():
        break
    features = resp.json().get('features', [])
    if not features:
        break

    for f in features:
        p = f['properties']
        if SOURCE and p.get('source') != SOURCE:
            continue
        all_events.append({
            'event_id':    str(p['eventid']),
            'storm_name':  p.get('name', ''),
            'source':      p.get('source', ''),
            'from_date':   p.get('fromdate', ''),
            'to_date':     p.get('todate', ''),
            'alert_level': p.get('alertlevel', ''),
        })

    print(f'Page {page}: {len(features)} events fetched')
    page += 1

df_events = pd.DataFrame(all_events)
print(f'\nTotal TCs found ({SOURCE or "all basins"}): {len(df_events)}')
df_events

Page 1: 100 events fetched
Page 2: 100 events fetched
Page 3: 90 events fetched

Total TCs found (NOAA): 52


,event_id,storm_name,source,from_date,to_date,alert_level
0,1001230,Tropical Cyclone MELISSA-25,NOAA,2025-10-21T15:00:00,2025-10-31T15:00:00,Red
1,1001168,Tropical Cyclone ERICK-25,NOAA,2025-06-16T21:00:00,2025-06-20T03:00:00,Red
2,1001122,Tropical Cyclone RAFAEL-24,NOAA,2024-11-03T21:00:00,2024-11-10T21:00:00,Orange
3,1001114,Tropical Cyclone OSCAR-24,NOAA,2024-10-19T15:00:00,2024-10-22T18:00:00,Orange
4,1001113,Tropical Cyclone NADINE-24,NOAA,2024-10-18T21:00:00,2024-10-20T15:00:00,Orange
5,1001111,Tropical Cyclone MILTON-24,NOAA,2024-10-05T15:00:00,2024-10-10T21:00:00,Orange
6,1001100,Tropical Cyclone JOHN-24,NOAA,2024-09-22T21:00:00,2024-09-27T21:00:00,Orange
7,1001101,Tropical Cyclone HELENE-24,NOAA,2024-09-23T15:00:00,2024-09-27T21:00:00,Red
8,1001067,Tropical Cyclone BERYL-24,NOAA,2024-06-28T21:00:00,2024-07-09T09:00:00,Red
9,1001066,Tropical Cyclone ALBERTO-24,NOAA,2024-06-17T21:00:00,2024-06-20T21:00:00,Orange


In [ ]:
# ── Helper: fetch national exposure for one event (latest episode) ────────────
def fetch_national_exposure(event_id):
    '''Return a list of country-row dicts for the latest episode of a TC event.
    Returns an empty list if the event has no impact data.'''
    try:
        props = requests.get(
            f'{GDACS_BASE}/events/getepisodedata',
            params={'eventtype': 'TC', 'eventid': event_id},
            timeout=30,
        ).json()['properties']
    except Exception as e:
        print(f'  [{event_id}] getepisodedata failed: {e}')
        return []

    last_ep_url = props.get('episodes', [{}])[-1].get('details', '')
    episode_id  = last_ep_url.split('episodeid=')[-1].split('&')[0] if last_ep_url else '?'

    resources = props.get('impacts', [{}])[0].get('resource', {})
    available_buffers = {k: v for k, v in resources.items() if k.startswith('buffer')}
    if not available_buffers:
        return []

    def scalars(row):
        return {s['name']: s['value'] for s in row['scalars']['scalar']}

    country_data = {}
    for buf, url in available_buffers.items():
        kt  = BUFFER_KT.get(buf, buf)
        col = f'pop_{kt}kt'
        try:
            datums = requests.get(url, timeout=30).json().get('datums', [])
        except Exception:
            continue
        country_datum = next((d for d in datums if d['alias'] == 'country'), None)
        if not country_datum:
            continue
        for row in country_datum.get('datum', []):
            sc   = scalars(row)
            iso3 = sc.get('ISO_3DIGIT')
            if not iso3:
                continue
            if iso3 not in country_data:
                country_data[iso3] = {
                    'event_id':     event_id,
                    'episode_id':   episode_id,
                    'iso3':         iso3,
                    'country_name': sc.get('CNTRY_NAME'),
                }
            # NOTE: Some storms have POP_AFFECTED_TEMP, which seems to consistently have a value of 0
            # We ignore the 0 in those cases and only populate the column if POP_AFFECTED is present and non-zero
            country_data[iso3][col] = int(sc['POP_AFFECTED']) if sc.get('POP_AFFECTED') else None

    return list(country_data.values())

In [14]:
# ── Fetch national exposure for every event ───────────────────────────────────
all_rows = []

for _, ev in df_events.iterrows():
    eid  = ev['event_id']
    name = ev['storm_name']
    print(f'Fetching {name} ({eid}) …', end=' ')
    rows = fetch_national_exposure(eid)
    for r in rows:
        r['storm_name']  = name
        r['source']      = ev['source']
        r['from_date']   = ev['from_date']
        r['alert_level'] = ev['alert_level']
    all_rows.extend(rows)
    print(f'{len(rows)} countries')

df = (
    pd.DataFrame(all_rows)
    .sort_values(['from_date', 'storm_name'], ascending=False)
    .reset_index(drop=True)
)

# Reorder columns
leading  = ['storm_name', 'event_id', 'episode_id', 'source', 'from_date', 'alert_level',
            'iso3', 'country_name']
pop_cols = sorted([c for c in df.columns if c.startswith('pop_')])
df = df[leading + pop_cols]

print(f'\nTotal rows : {len(df)}')
print(f'Storms     : {df["storm_name"].nunique()}')
print(f'Countries  : {df["iso3"].nunique()}')
df

Fetching Tropical Cyclone MELISSA-25 (1001230) … 9 countries
Fetching Tropical Cyclone ERICK-25 (1001168) … 1 countries
Fetching Tropical Cyclone RAFAEL-24 (1001122) … 3 countries
Fetching Tropical Cyclone OSCAR-24 (1001114) … 4 countries
Fetching Tropical Cyclone NADINE-24 (1001113) … 5 countries
Fetching Tropical Cyclone MILTON-24 (1001111) … 4 countries
Fetching Tropical Cyclone JOHN-24 (1001100) … 1 countries
Fetching Tropical Cyclone HELENE-24 (1001101) … 1 countries
Fetching Tropical Cyclone BERYL-24 (1001067) … 13 countries
Fetching Tropical Cyclone ALBERTO-24 (1001066) … 4 countries
Fetching Tropical Cyclone OTIS-23 (1001028) … 1 countries
Fetching Tropical Cyclone NORMA-23 (1001024) … 1 countries
Fetching Tropical Cyclone LIDIA-23 (1001019) … 1 countries
Fetching Tropical Cyclone IDALIA-23 (1001000) … 1 countries
Fetching Tropical Cyclone FRANKLIN-23 (1000996) … 4 countries
Fetching Tropical Cyclone HILARY-23 (1000993) … 2 countries
Fetching Tropical Cyclone LISA-22 (1000944) 

,storm_name,event_id,episode_id,source,from_date,alert_level,iso3,country_name,pop_34kt,pop_64kt
0,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,CAN,Canada,581439.0,1813.0
1,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,BMU,Bermuda,16570.0,NaN
2,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,USA,United States,7959.0,NaN
3,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,SPM,St. Pierre & Miquelon,5676.0,NaN
4,Tropical Cyclone MELISSA-25,1001230,41,NOAA,2025-10-21T15:00:00,Red,BHS,The Bahamas,14029.0,2838.0
...,...,...,...,...,...,...,...,...,...,...
209,Tropical Cyclone ERIKA-15,1000203,19,NOAA,2015-08-25T03:00:00,Orange,KNA,St. Kitts & Nevis,30935.0,NaN
210,Tropical Cyclone ERIKA-15,1000203,19,NOAA,2015-08-25T03:00:00,Orange,ATG,Antigua & Barbuda,88267.0,NaN
211,Tropical Cyclone ERIKA-15,1000203,19,NOAA,2015-08-25T03:00:00,Orange,MSR,Montserrat,4663.0,NaN
212,Tropical Cyclone ERIKA-15,1000203,19,NOAA,2015-08-25T03:00:00,Orange,GLP,Guadeloupe,378381.0,NaN


In [15]:
# ── Export ────────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved → {OUTPUT_CSV}')

Saved → gdacs_historical_national_exposure.csv
